# Forward Backward matching debug

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
from dotenv import find_dotenv, load_dotenv

from n23_fluctuations_ragland import tools

_ = load_dotenv(find_dotenv())

%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"

with open(fn, "rb") as f:
    ray_run = pickle.load(f)

rays = ray_run["rays"]
rays_back = ray_run["rays_back"]
eigs = ray_run["eigs"]
eigs_back = ray_run["eigs_back"]
envi = ray_run["envi"]
envi_back = ray_run["envi_back"]


In [ ]:
plt.close()

fig = plt.figure(figsize=(10,4))

ray_id = '-7'

ray_id_pr, (sign, number, suffix) = tools.parse_ray_id(ray_id)

# Determine backward ray_id (flip sign if even number of turning points)
if number % 2 != 0:
    ray_id_back = ray_id
else:
    sign_back = "+" if sign == "-" else "-"
    ray_id_back = f"{sign_back}{number}{suffix}"

# Convert backward ray_id to pr format
ray_id_pr_back, _ = tools.parse_ray_id(ray_id_back)

ray_idx_forward = np.where(eigs.ray_id[0] == ray_id_pr)[0]
ray_idx_backward = np.where(eigs_back.ray_id[0] == ray_id_pr_back)[0]


fig.suptitle(ray_id_pr)


plt.subplot(1,2,1)

rays.plot_depth_v_angle(s=4, lw=0)
plt.vlines(eigs.launch_angles[0][ray_idx_forward], -5000,0, colors='k')
plt.vlines(-eigs_back.received_angles[0][ray_idx_backward], -5000,0, colors='r')

plt.title('forward rays')

plt.hlines(-1000, -30,30, colors='k')

plt.subplot(1,2,2)
rays_back.plot_depth_v_angle(s=4, lw=0)
plt.vlines(eigs.received_angles[0][ray_idx_forward], -5000,0, colors='k')
plt.vlines(-eigs_back.launch_angles[0][ray_idx_backward], -5000,0, colors='r')

plt.title('backward rays')
plt.hlines(-2300, -30,30, colors='k')

plt.tight_layout()

plt.show()

print(eigs.launch_angles[0][ray_idx_forward], eigs.received_angles[0][ray_idx_forward])
print(-eigs_back.received_angles[0][ray_idx_backward], -eigs_back.launch_angles[0][ray_idx_backward])